# Attention language model

Start from the same Tiny Shakespeare token stream as the bigram model, then run explicit multi-head causal self-attention on the embedded batch.

In [9]:
import random
import sys
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.dataset import get_batch, load_tiny_shakespeare_tokens, split_token_stream

## Prepare token streams and batches

In [10]:
tokens, vocab, _ = load_tiny_shakespeare_tokens(repo_root / "data")
train_tokens, validation_tokens = split_token_stream(tokens)

vocab_size = len(vocab)
block_size = 8
batch_size = 32
n_embd = 32
num_heads = 2
head_size = n_embd // num_heads

random.seed(42)
x_batch, y_batch = get_batch("train", train_tokens, validation_tokens, block_size, batch_size)
x_batch = torch.tensor(x_batch, dtype=torch.long)
y_batch = torch.tensor(y_batch, dtype=torch.long)

## Embed the current batch

In [11]:
token_embedding_table = nn.Embedding(vocab_size, n_embd)
x = token_embedding_table(x_batch)

assert x.shape == (batch_size, block_size, n_embd)
tuple(x.shape)

(32, 8, 32)

## Multi-head causal self-attention

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, d_model, d_k, d_v):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v

        self.w_qs = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_ks = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_vs = nn.Linear(d_model, n_head * d_v, bias=False)
        self.output_projection = nn.Linear(n_head * d_v, d_model, bias=False)

    def forward(self, query, key, value):
        d_k, d_v, n_head = self.d_k, self.d_v, self.n_head
        B, len_q, _ = query.shape
        _, len_k, _ = key.shape
        _, len_v, _ = value.shape

        q = self.w_qs(query).view(B, len_q, n_head, d_k)
        k = self.w_ks(key).view(B, len_k, n_head, d_k)
        v = self.w_vs(value).view(B, len_v, n_head, d_v)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # scaled dot-product attention
        # q @ k^T: (B, h, T, d_k) @ (B, h, d_k, T)
        #          -> (B, h, T, T)
        scores = q @ k.transpose(-2, -1)
        scores = scores / (d_k**0.5)

        # causal mask
        causal_mask = torch.triu(
            torch.ones(len_q, len_k, device=scores.device, dtype=torch.bool), diagonal=1
        )
        scores = scores.masked_fill(causal_mask, float("-inf"))

        # normalize, then retrieve values
        weights = torch.softmax(scores, dim=-1)  # (B, h, T, T)
        output = weights @ v  # (B, h, T, d_v)

        # put heads beside each other again
        output = output.transpose(1, 2).contiguous()  # (B, T, h, d_v)
        output = output.view(B, len_q, n_head * d_v)  # (B, T, h*d_v)
        output = self.output_projection(output)  # (B, T, C)

        return output, weights

class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.w_2(F.relu(self.w_1(x)))

x = torch.randn(batch_size, block_size, n_embd)
mha = MultiHeadAttention(
    n_head=num_heads, d_model=n_embd, d_k=head_size, d_v=head_size
)
out, weights = mha(x, x, x)

assert out.shape == x.shape
assert weights.shape == (batch_size, num_heads, block_size, block_size)
tuple(out.shape)

(32, 8, 32)

## Next step

Wrap this attention module in a minimal transformer block and verify that both residual paths preserve `(B, T, C)`.

In [13]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, d_k, d_v, d_ff):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(n_head, d_model, d_k, d_v)

        self.ln2 = nn.LayerNorm(d_model)
        self.ff = PositionWiseFeedForward(d_model, d_ff)

    def forward(self, x):
        # attention residual
        z = self.ln1(x)
        attn_out, weights = self.attn(z, z, z)
        x = x + attn_out

        # feed-forward residual
        x = x + self.ff(self.ln2(x))

        return x, weights

In [14]:
# test the above

# In[1]:

block = TransformerBlock(
    d_model=n_embd,
    n_head=num_heads,
    d_k=head_size,
    d_v=head_size,
    d_ff=4 * n_embd,
)

out, weights = block(x)

print(x.shape)
print(out.shape)
print(weights.shape)

torch.Size([32, 8, 32])
torch.Size([32, 8, 32])
torch.Size([32, 2, 8, 8])
